In [1]:
%%capture
%pip install gradio
%pip install langchain-text-splitters
%pip install langchain-community
%pip install chromadb
#%pip install --force-reinstall numpy==1.26.4
%pip install jq

In [2]:
####### Start Ollama server #########

!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

# Start the server in the background
process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Give the server a few seconds to initialize
time.sleep(5)
print("Ollama server is running in the background!")
!ollama pull llama3

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 10 not upgraded.
>>> Cleaning up old version at /usr/local/lib/

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain_community.document_loaders import JSONLoader
import json
import jq
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
import gradio as gr
import logging
import sys
import warnings

# To suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
warnings.warn = warn
warnings.filterwarnings('ignore')

def process_json_file(file_path):
  if file_path is None:
    raise gr.Error("Please upload a JSON file first.")

  try:
    with open(file_path, "r", encoding="utf-8") as f:
      data = json.load(f)

    # Perform your JSON processing logic here (e.g., validate keys, transform data)
    # If everything is valid:
    gr.Info("JSON processed successfully!")
    return (
      f"Success! Loaded JSON with {len(data) if isinstance(data, (dict, list)) else 'valid'} entries."
    )

  except json.JSONDecodeError as e:
    raise gr.Error(f"Invalid JSON format: {str(e)}")
  except Exception as e:
    raise gr.Error(f"An error occurred while processing: {str(e)}")

## Document loader
def document_loader(file):
    if file is None:
        logging.info( "Please upload a valid JSON file.")

    loader = JSONLoader(
        file_path=file,
        jq_schema='.[]', # Adjust the jq_schema based on your JSON structure
        text_content=False
    )
    loaded_document = loader.load()
    return loaded_document

## Text splitter
def text_splitter(data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=20,
        length_function=len,
    )
    chunks = text_splitter.split_documents(data)
    return chunks


## Token embedding
def hf_embedding():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    model_kwargs = {'device': 'cpu'}
    encode_kwargs = {'normalize_embeddings': False}

    embedding_model = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs=model_kwargs,
        encode_kwargs=encode_kwargs
    )
    return embedding_model

## Initialize vector db
def vector_database(chunks):
    embedding_model = hf_embedding()
    vectordb = Chroma.from_documents(chunks, embedding_model)
    return vectordb

#Create vector db
def create_vector_db(file):
    if file is None:
        logging.info("Please upload a valid JSON file.")
    splits = document_loader(file)
    chunks = text_splitter(splits)
    vectordb = vector_database(chunks)
    return vectordb.as_retriever()

## Query Chain
def retriever_qa(file_input, query):
    retriever_state = create_vector_db(file_input)
    if retriever_state is None:
        logging.error("The provided retriever object is None.")
        return (f"iThe provided retriever object is None.")

    try:
      # 1. Define custom prompt template
      template = """You are an expert fact-extraction system. You must output EXACTLY the correct answer from the provided context. Do NOT use conversational filler, do NOT add introductory text like 'According to the context', and do NOT output full sentences. Only output the exact answer.
      Context: {context}
      Question: {question}
      Answer:"""

      QA_CHAIN_PROMPT = PromptTemplate(
          input_variables=["context", "question"],
          template=template,
      )

      llm = Ollama(model="llama3")

      qa = RetrievalQA.from_chain_type(
          llm=llm,
          chain_type="stuff",
          retriever=retriever_state,
          return_source_documents=True,
          chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}
      )



      response = qa.invoke(query)
      if isinstance(response, dict):
          answer = response.get("result", "").strip()
          docs = response.get("source_documents", [])
      else:
          answer = str(response).strip()
          docs = retriever_state.get_relevant_documents(query)
      context = "\n".join([doc.page_content for doc in docs])
      return answer

    except Exception as e:
      logging.error(f"Error during chain invocation: {e}")
      return "The LLM could not generate an answer from the retrieved context."




# Gradio Interface using State
with gr.Blocks() as rag_application:
    gr.Markdown("# AI: Upload a JSON document and ask any question.")

    with gr.Row():
        file_input = gr.File(label="Upload JSON File", file_count="single", file_types=['.json'], type="filepath")
        process_button = gr.Button("Process JSON")
    output_status = gr.Textbox(label="Processing Status", interactive=False)

    # Connect the button click event
    process_button.click(
        fn=process_json_file, inputs=file_input, outputs=output_status
    )

    with gr.Row():
        query_input = gr.Textbox(label="Input Query", lines=2, placeholder="Type your question here...")
        output_box = gr.Textbox(label="Output")

    with gr.Row():
        submit_query_btn = gr.Button("Submit Question")

    submit_query_btn.click(fn=retriever_qa, inputs=[file_input, query_input], outputs=output_box)


# Launch the app
rag_application.launch(server_name="127.0.0.1",share=True,debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0f6e7c1249fafae1dc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://0f6e7c1249fafae1dc.gradio.live


In [ ]:
rag_application.close()